# Kimodo 3D Human Motion Generation on Kaggle

This notebook installs Kimodo, generates a human motion sequence, and launches the Soma Viser viewer.


## 1. Configure Runtime Storage

Set temporary directories for model downloads, caches, and generated outputs.


In [ ]:
import os
from pathlib import Path

# Temporary/high-volume storage
TMP_ROOT = Path("/tmp/kimodo_runtime")
HF_HOME = TMP_ROOT / "huggingface"
HF_HUB_CACHE = HF_HOME / "hub"
TRANSFORMERS_CACHE = HF_HOME / "transformers"
TORCH_HOME = TMP_ROOT / "torch"

for path in [HF_HOME, HF_HUB_CACHE, TRANSFORMERS_CACHE, TORCH_HOME]:
    path.mkdir(parents=True, exist_ok=True)

# Keep final results in Kaggle's persistent working directory
OUTPUT_DIR = Path("/kaggle/working/kimodo_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HF_HUB_CACHE"] = str(HF_HUB_CACHE)
os.environ["TRANSFORMERS_CACHE"] = str(TRANSFORMERS_CACHE)
os.environ["TORCH_HOME"] = str(TORCH_HOME)

# Use GPU 0 for Kimodo.
# GPU 1 remains available for another independent process.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Avoid loading the large text encoder onto the T4.
os.environ["TEXT_ENCODER_DEVICE"] = "cpu"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("HF cache:", HF_HUB_CACHE)
print("Output directory:", OUTPUT_DIR)

## 2. Clone the Kimodo Repository

Download the Kimodo source code into the Kaggle working directory.


In [ ]:
%cd /kaggle/working

!rm -rf kimodo
!git clone https://github.com/nv-tlabs/kimodo.git
%cd /kaggle/working/kimodo

## 3. Install Build Dependencies

Upgrade pip and install the required build tools and CMake version.


In [ ]:
!pip install -q --upgrade pip
!pip install -q --upgrade "setuptools<82" "cmake>=3.28"

## 4. Install Python Dependencies

Install PyTorch-related and Hugging Face dependencies required by Kimodo.


In [ ]:
!pip install -q --upgrade "torchao>=0.16.0" peft transformers huggingface_hub

## 5. Install Kimodo

Install the local Kimodo package in editable mode.


In [ ]:
!pip install -e . --no-build-isolation

## 6. Authenticate with Hugging Face

Log in to Hugging Face. Before running this cell, accept the access agreement for `meta-llama/Meta-Llama-3-8B-Instruct` on Hugging Face.

In [ ]:
from huggingface_hub import login

login()

## 7. Verify the Environment

Check the installed Kimodo package, PyTorch version, and CUDA availability.


In [ ]:
import torch
import kimodo

print("Kimodo:", kimodo.__file__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Visible GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

## 8. Configure Multiprocessing

Use the file-system sharing strategy to reduce multiprocessing shared-memory issues.


In [ ]:
import torch.multiprocessing as mp

mp.set_sharing_strategy("file_system")
print("Sharing strategy:", mp.get_sharing_strategy())

## 9. Generate Human Motion

Run Kimodo with the selected prompt and save the generated motion file.


In [ ]:
%cd /kaggle/working/kimodo

!TEXT_ENCODER_DEVICE=cpu \
  HF_HOME=/tmp/kimodo_runtime/huggingface \
  HF_HUB_CACHE=/tmp/kimodo_runtime/huggingface/hub \
  kimodo_gen "A person doing frontflip." \
  --model Kimodo-SOMA-RP-v1 \
  --duration 5.0 \
  --diffusion_steps 25 \
  --seed 42 \
  --output /kaggle/working/kimodo_outputs/frontflip

## 10. Convert the Generated NPZ Motion to BVH

Convert the generated Kimodo .npz motion file into .bvh format for use with Soma Viser.

In [ ]:
from pathlib import Path
import subprocess

# Update these paths with your input NPZ file and desired output BVH filename.
input_npz = Path("/kaggle/working/kimodo_outputs/frontflip.npz")  # Update input file path
output_bvh = Path("/kaggle/working/kimodo_outputs/frontflip.bvh")  # Update output file path

subprocess.run([
    "kimodo_convert",
    str(input_npz),
    str(output_bvh),
], check=True)

print("Created:", output_bvh)

## 11. Install Soma Viser

Clone the Soma Viser repository and install it in editable mode.


In [ ]:
%cd /kaggle/working

!git clone https://github.com/simeon-ned/soma_viser.git
%cd /kaggle/working/soma_viser

!pip install -e .

## 12. Patch Soma Viser BVH Import Compatibility

Updates the BVH loader import to match the installed soma_retargeter package.

In [ ]:
from pathlib import Path

motion_file = Path(
    "/kaggle/working/soma_viser/src/soma_viser/core/motion.py"
)

text = motion_file.read_text()

text = text.replace(
    "from soma_retargeter.assets.bvh import load_bvh",
    "from soma_retargeter.io.bvh import load_bvh",
)

motion_file.write_text(text)

print("Soma Viser compatibility patch applied.")

## 13. Copy the Motion File

Copy the generated BVH file into Soma Viser’s motions directory so the viewer can load it.

In [ ]:
from pathlib import Path
import shutil

src = Path("/kaggle/working/kimodo_outputs/frontflip.bvh")  # Update BVH file path
dst_dir = Path("/kaggle/working/soma_viser/motions")
dst_dir.mkdir(parents=True, exist_ok=True)

shutil.copy2(src, dst_dir / src.name)

print("Copied:", dst_dir / src.name)

## 14. Start Soma Viser

Launch Soma Viser and create a Cloudflare Tunnel for public access.

In [ ]:
import subprocess
import time
from pathlib import Path

PORT = "8090"
MOTIONS_DIR = "/kaggle/working/soma_viser/motions"

viewer = subprocess.Popen(
    [
        "soma-viser",
        "--motions-dir", MOTIONS_DIR,
        "--port", PORT,
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

time.sleep(5)

if viewer.poll() is not None:
    print("❌ Soma Viser exited.")
    print(viewer.stdout.read())
else:
    print("✅ Soma Viser started on port 8090.")

    cloudflared = Path("/tmp/cloudflared")

    if not cloudflared.exists():
        subprocess.run([
            "wget", "-q",
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            "-O", str(cloudflared),
        ], check=True)
        cloudflared.chmod(0o755)

    print("Starting Cloudflare Tunnel...")

    subprocess.run([
        str(cloudflared),
        "tunnel",
        "--url", f"http://127.0.0.1:{PORT}",
    ])